In [1]:
import sys
import os
import time
import numpy as np
import matplotlib.pyplot as plt

# Thêm thư mục src vào hệ thống import
sys.path.append(os.path.abspath("../src"))

from dataset import load_and_preprocess_mnist
from metrics import compute_classification_metrics
from models.naive_bayes import NaiveBayes
from models.softmax_regression import SoftmaxRegression
from models.mlp import MLPFromScratch
from models.ensemble import RandomForestFromScratch

In [2]:
# Nạp dữ liệu đã tiền xử lý chuẩn hóa
X_train, y_train, X_val, y_val, X_test, y_test = load_and_preprocess_mnist()

print(f"Tap Huan luyen (Train)     : {X_train.shape}, nhan: {y_train.shape}")
print(f"Tap Tham dinh (Validation) : {X_val.shape}, nhan: {y_val.shape}")
print(f"Tap Kiem thu (Test)        : {X_test.shape}, nhan: {y_test.shape}")
print(f"Mien gia tri pixel         : Min = {X_train.min():.2f}, Max = {X_train.max():.2f}")

100.0%
100.0%
100.0%
100.0%


Tap Huan luyen (Train)     : (50000, 784), nhan: (50000,)
Tap Tham dinh (Validation) : (10000, 784), nhan: (10000,)
Tap Kiem thu (Test)        : (10000, 784), nhan: (10000,)
Mien gia tri pixel         : Min = 0.00, Max = 1.00


In [3]:
models = {
    "Naive Bayes (Multinomial)": NaiveBayes(alpha=0.01),
    "Softmax Regression": SoftmaxRegression(lr=0.1, max_epochs=15, batch_size=128, lam=1e-3),
    "MLP from Scratch": MLPFromScratch(hidden_sizes=[128, 64], lr=0.05, momentum=0.9, max_epochs=15, batch_size=128),
    "Random Forest (Bagging)": RandomForestFromScratch(n_estimators=30, max_depth=15, max_features="sqrt")
}

results = {}

for name, model in models.items():
    print(f"\n==========================================")
    print(f"Dang huan luyen: {name}...")
    
    # Đo thời gian huấn luyện
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0
    
    # Đo thời gian dự đoán trên tập Test
    t1 = time.time()
    y_pred = model.predict(X_test)
    infer_time = time.time() - t1
    
    # Tính các chỉ số đánh giá bằng module metrics.py
    metrics = compute_classification_metrics(y_test, y_pred, n_classes=10)
    
    results[name] = {
        "Accuracy": metrics["accuracy"],
        "Macro Precision": metrics["precision_macro"],
        "Macro Recall": metrics["recall_macro"],
        "Macro F1": metrics["f1_macro"],
        "Train Time (s)": train_time,
        "Infer Time (s)": infer_time,
        "Confusion Matrix": metrics["confusion_matrix"]
    }
    
    print(f"Hoan tat! Accuracy: {metrics['accuracy']*100:.2f}% | F1: {metrics['f1_macro']:.4f} | Thoi gian: {train_time:.1f}s")


Dang huan luyen: Naive Bayes (Multinomial)...
Hoan tat! Accuracy: 83.60% | F1: 0.8345 | Thoi gian: 0.1s

Dang huan luyen: Softmax Regression...
Hoan tat! Accuracy: 92.03% | F1: 0.9192 | Thoi gian: 6.7s

Dang huan luyen: MLP from Scratch...
Hoan tat! Accuracy: 97.92% | F1: 0.9791 | Thoi gian: 18.1s

Dang huan luyen: Random Forest (Bagging)...
Hoan tat! Accuracy: 95.89% | F1: 0.9585 | Thoi gian: 31.7s
